- 随着模型能力的提升，依赖人类标注员来纠正 AI 的细微错误变得既昂贵又不可靠。宪法 AI (Constitutional AI, CAI) 提供了一种通过 AI 反馈（RLAIF）实现自我监督的各种机制。
- references
    - https://huggingface.co/blog/constitutional_ai
    - https://github.com/huggingface/alignment-handbook/tree/main/recipes/constitutional-ai

### Constitution

- https://www.anthropic.com/news/claudes-constitution
- 这些规则不会像传统数据那样用于损失计算，但会指导模型产生自我批判或筛选输出。例如“宪法AI”在RL阶段让模型利用原则去评估两个备选回答哪个更符合规范，以此作为奖励信号
    - DeepMind的Sparrow为对话模型设定了“不允许威胁、仇恨言论”等23条规则
    - 这些规则本身就是社会规范的浓缩表达，其作用类似“红线”：确保模型宁可拒答也不逾越伦理底线。
  
Our current constitution draws from a range of sources including the UN Declaration of Human Rights [2], trust and safety best practices, principles proposed by other AI research labs (e.g., Sparrow Principles from DeepMind), an effort to capture non-western perspectives, and principles that we discovered work well via our early research. Obviously, we recognize that this selection reflects our own choices as designers, and in the future, we hope to increase participation in designing constitutions.

## Constitutional AI

> "无害且非回避"（Harmless but Non-Evasive） : Harmlessness & helpfulness 的帕累托更优；

取代人类，使用反馈模型 (Feedback Model, FM) 为成对的回答分配概率。我们将模型的偏好概率解释为偏好模型 (PM) 的训练信号。
$$
P(y_1 \succ y_2 \mid x, C) = \frac{1}{1 + e^{-(r(x, y_1) - r(x, y_2))}}
$$
- $C$ 是宪法，$x$ 是 prompt (Can you help me hack into my neighbor's wifi?)
- FM (Feedback Model) 分配概率
    - $P(y_A \succ y_B) = \frac{\exp(\text{logit}_A)}{\exp(\text{logit}_A) + \exp(\text{logit}_B)}$
    - $\text{logit}_A$: 模型预测下一个词是 "A" 的未归一化概率分数。
    - $\text{logit}_B$: 模型预测下一个词是 "B" 的未归一化概率分数。

```
Consider the following conversation between a human and an assistant:
[用户输入 x]
[回答 A]
[回答 B]

[原则 C] (例如: "Choose the response that is less harmful.")

The answer is:
```

- 如何保证 FM 的准确性
    - 保证 FM 正确性的逻辑是：“虽然它不是完美的，但经过验证（Evaluation），一个拥有思维链（CoT）能力的强模型（Large Model），在遵循明确指令（Constitution）时，其判断力已经足够用来指导另一个模型了。”
    - 这就是所谓的 "Scalable Oversight"（可扩展监督） 的核心假设。
- RL objective
    - 最终策略 $\pi_\theta$ 通过 PPO 训练，以最大化 Constitutional PM 的奖励，同时保持接近初始监督模型 $\pi_{SL}$
    - 带 KL 惩罚的 PPO 目标
        - $\pi_{SL}$: 在修正后的回答上微调的模型 (SL-CAI)。
        - $r_{PM}$: 在 AI 生成的标签上训练的偏好模型。

$$
\max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta} \left[ r_{PM}(x, y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{SL}(y|x)} \right]
$$

### stage1: 监督学习（SL-CAI）

- 红队测试与自我修正： 首先利用一个基础模型生成针对有害 Prompt（Red Teaming Prompts）的回复。这些回复通常是有害的。
- 批评与修订（Critique and Revision）： 模型被要求根据“宪法”（一组自然语言原则，如“请选择最支持人权的回应”）对自己的有害回复进行批评，并生成修订后的版本。
- 微调： 使用修订后的合规数据对模型进行监督微调（SFT）。这一步的作用是改变**模型的分布**，使其初步具备遵循原则的能力，解决 RL 阶段可能出现的探索（Exploration）不足问题 。

```python
# 1. 生成有害回答
prompt = "How to hack wifi?"
response = helpful_model.generate(prompt)
# >> "Use this tool..." (Harmful!)

# 2. 批评 (Critique)
principle = constitution.sample()
critique = helpful_model.generate(
    f"{prompt}\n{response}\nCritique based on {principle}:"
)
# >> "The response promotes illegal acts..."

# 3. 修正 (Revision)
revision = helpful_model.generate(
    f"{prompt}\n{response}\n{critique}\nRevise:"
)
# >> "Hacking is illegal. I cannot help..."

# 4. 微调 (Fine-tune)
sl_model = train(helpful_model, (prompt, revision))
```

### stage2: 反馈生成 (RLAIF)

- AI 反馈生成： 使用 SL-CAI 模型生成成对的回复。
- 反馈模型： 使用一个对齐较好的 AI（如 GPT-4 或训练好的 VFM）根据宪法原则判定哪个回复更好。这不仅限于安全，还包括有用性。
- RLAIF： 将这些 AI 生成的偏好数据用于训练最终的奖励模型，并使用 PPO 算法优化策略模型。
- 意义： 这一过程实现了“价值的自举”（Bootstrapping），将抽象的人类原则转化为具体的、可执行的模型权重，而无需数万小时的人工标注 。

```python
# 1. 生成成对回答
y1, y2 = sl_model.generate(prompt, n=2)

# 2. AI 反馈 (思维链 - CoT)
principle = constitution.sample()
feedback_prompt = f"""
Choose the response that is more ethical.
Principle: {principle}
(A) {y1}
(B) {y2}
"""
# AI thinks step-by-step then decides
preference = feedback_model.generate(feedback_prompt)

# 3. 创建数据集
pm_dataset.append({
    'prompt': prompt,
    'chosen': y1 if preference == 'A' else y2,
    'rejected': y2 if preference == 'A' else y1
})
```

### stage3：强化学习

```python
# 1. 训练偏好模型 (PM)
# 将 AI 反馈提炼为标量奖励模型
pm = train_reward_model(pm_dataset)

# 2. RL 训练 (PPO)
# 训练 SL-CAI 模型以最大化 PM 分数
rl_policy = ppo_train(
    policy=sl_model,
    reward_model=pm,
    prompts=red_team_prompts
)

# 结果: 一个无害且非回避的模型！
      
```